# 3. Compare independent exports and join a separate catalogue

This tutorial uses two independently identified exports of **one unchanged saved Calliope execution**. The future-scenario descriptions, study context and catalogue publisher are synthetic. No alternative future is simulated and no numerical result is changed. The expected zero differences test the representation and query rules; they are not evidence about the effect of a scenario.

The existing [Python demonstration](../examples/annotations/compare_exports.py) supplies the complete preparation logic. This notebook calls it, then inspects the graphs, query answers and refusal checks in Python. It runs independently of the other notebooks and writes to `build/notebooks/03_comparison_and_integration/`.

In [1]:
from pathlib import Path
import json
from decimal import Decimal

import pandas as pd
from IPython.display import display
from calliomapper import Translator
from calliomapper.queries import run_query
from calliomapper.vocabulary import OC

REPO_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
    and (path / "examples" / "calliope-0.7.0-glpk").is_dir()
)
EXAMPLES = REPO_ROOT / "examples" / "calliope-0.7.0-glpk"
NATIONAL = EXAMPLES / "national_scale" / "results_directory"
OUT = REPO_ROOT / "build" / "notebooks" / "03_comparison_and_integration"
OUT.mkdir(parents=True, exist_ok=True)


def answer_table(result):
    """Keep RDF numeric values and represent unavailable values as None."""
    return pd.DataFrame(
        [[term.toPython() if term is not None else None for term in row] for row in result],
        columns=[str(variable) for variable in result.vars],
    )


import runpy
from rdflib import Dataset

ANNOTATIONS = EXAMPLES / "annotations" / "national.yaml"

## Prepare the two exports using the maintained Python example

The demonstration first creates exports with independent namespaces. It then supplies a common study, a reference and an alternative future description, an explicit correspondence between the CCGT components, and supported unit interpretations. A separately named catalogue graph supplies a fictional publisher for their shared dataset revision.

`runpy` loads the existing example file without invoking its command-line entry point. Keeping this logic in one place lets the notebook demonstrate the same implementation used by the executable example.

In [2]:
demo = runpy.run_path(str(REPO_ROOT / "examples" / "annotations" / "compare_exports.py"))
report = demo["build_demo"](OUT, results_directory=NATIONAL, annotations_path=ANNOTATIONS)
assert all(report["checks"].values())
assert report["checks"]["same_saved_execution_digest"]
assert report["checks"]["distinct_component_identifiers"]

combined = Dataset(default_union=True)
combined.parse(OUT / "combined.nq", format="nquads")
display(pd.DataFrame([{
    "CQ-06 comparison rows": report["comparison_rows"],
    "CQ-07 integration rows": report["integration_rows"],
    "same saved execution": report["checks"]["same_saved_execution_digest"],
    "distinct component identifiers": report["checks"]["distinct_component_identifiers"],
}]))

,CQ-06 comparison rows,CQ-07 integration rows,same saved execution,distinct component identifiers
0,1,38,True,True


## Inspect the capacity comparison

Competency question 6 requires more than equal technology names. This example supplies an explicit component correspondence, compatible units and time coverage, and the study/scenario relationships required by the query. The resulting row compares the effective capacity bound and the calculated capacity.

Both exports reuse the same numerical execution, so the two differences must be zero. Separate graph identifiers do not, by themselves, mean that two simulations were performed.

In [3]:
comparison = answer_table(run_query(combined, "cq06"))
assert len(comparison) == 1
assert comparison.iloc[0]["inputDifference"] == Decimal("0")
assert comparison.iloc[0]["resultDifference"] == Decimal("0")
assert comparison.iloc[0]["runA"] != comparison.iloc[0]["runB"]
display(comparison[["inputVariable", "inputA", "inputB", "inputDifference",
                    "resultA", "resultB", "resultDifference", "unitLabel"]])

,inputVariable,inputA,inputB,inputDifference,resultA,resultB,resultDifference,unitLabel
0,flow_cap_max,30000.0,30000.0,0.0,30000.0,30000.0,0.0,kW


## Query shared information across the exports and catalogue

Competency question 7 joins the exports to a separately identified catalogue through an explicit dataset-revision identifier. Its SPARQL query uses OEO, BFO, PROV-O and Dublin Core terms over asserted facts; it does not depend on inferred triples or OntoCal-specific predicates.

The compact table below uses labels for the two namespaces. The complete identifiers remain in `integration`. Both execution representations must remain distinguishable, while the shared dataset revision connects them to the catalogue.

In [4]:
integration = answer_table(run_query(combined, "cq07"))
assert integration["run"].nunique() == 2
assert integration["export"].nunique() == 2
assert integration["dataset"].nunique() == 1
assert integration["catalogue"].nunique() == 1

namespaces = {
    label: json.loads((OUT / f"source-{label.lower()}.nq.manifest.json").read_text())["namespace"]
    for label in ("A", "B")
}

def export_label(identifier):
    return next(f"Export {label}" for label, namespace in namespaces.items()
                if str(identifier).startswith(namespace))

integration_view = integration.assign(
    export=integration["export"].map(export_label),
    run=integration["run"].map(lambda identifier: export_label(identifier) + " representation of the saved run"),
)
integration_summary = integration_view.groupby(["export", "run"], as_index=False).agg(
    result_records=("result", "nunique"),
    dataset_revisions=("dataset", "nunique"),
    catalogue_graphs=("catalogue", "nunique"),
)
print("Shared dataset revision:", integration["dataset"].iloc[0])
display(integration_summary)

Shared dataset revision: https://example.org/calliomapper/tutorial/dataset/capacity/revision-2


,export,run,result_records,dataset_revisions,catalogue_graphs
0,Export A,Export A representation of the saved run,19,1,1
1,Export B,Export B representation of the saved run,19,1,1


## Check that missing or incompatible information prevents comparison

The Python demonstration also creates temporary, deliberately altered copies in memory. Removing a required unit, changing the unit, or changing the represented time coverage must remove the comparison answer. Those altered copies are not saved as valid exports.

Passing these checks supports the implemented query conditions. It does not validate the fictional scenario narratives or establish causal conclusions from the compared numbers.

In [5]:
refusal_checks = {
    "different units prevent comparison": report["checks"]["different_units_refused"],
    "a missing unit prevents comparison": report["checks"]["missing_unit_refused"],
    "different time coverage prevents comparison": report["checks"]["different_horizon_refused"],
    "both export run identifiers remain available": report["checks"]["integration_retains_both_export_run_identifiers"],
}
assert all(refusal_checks.values())
display(pd.DataFrame(refusal_checks.items(), columns=["check", "passed"]))

,check,passed
0,different units prevent comparison,True
1,a missing unit prevents comparison,True
2,different time coverage prevents comparison,True
3,both export run identifiers remain available,True


## Inspect or extend the demonstration

The output directory contains both source-only exports, both annotated exports, the YAML annotations used for each, the catalogue graph, the combined graph, CSV query answers and `checks.json`. To inspect the supplied pairing or scenario text, open `annotations-a.yaml` or `annotations-b.yaml` there. The saved numerical inputs have not been edited.

For a real comparison, use independently documented saved executions and supply evidence for the study, scenario relationships, component correspondence, units and time coverage. The [question requirements](../docs/questions.md) state what each query needs and how missing information affects its answers.